In [ ]:
import os, glob, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

tf.keras.utils.set_random_seed(42)
DATA_DIR = '/kaggle/input/datasets/m8nabdou/server-workload-traces'
WIN, STEPS_DAY = 12, 288          # 1-hour look-back, 288 x 5 min = 1 day
COLS = ['ts','cores','cap','cpu_mhz','cpu','memcap','mem','dr','dw','nr','nt']

# ---------- 1. Load data ----------
csvs = glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True)
if not csvs:
    for z in glob.glob(f'{DATA_DIR}/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall('/kaggle/working/data')
    csvs = glob.glob('/kaggle/working/data/**/*.csv', recursive=True)
if not csvs:                                       # local fallback
    csvs = glob.glob('/home/claude/data/*.csv')
print(len(csvs), 'VM files found')

frames = []
for f in csvs:
    d = pd.read_csv(f); d.columns = COLS
    d.index = pd.to_datetime(d.ts, unit='s'); d = d.sort_index()
    frames.append(d[['cpu_mhz','cap','mem','dr','dw','nr','nt']].resample('5min').mean())

agg = pd.concat(frames).groupby(level=0).sum(min_count=1).dropna()   # total of all VMs
agg['load'] = 100 * agg.cpu_mhz / agg.cap
print(agg.shape, agg.index.min(), '->', agg.index.max())

# ---------- 2. Features ----------
d = agg[['load','mem','dr','dw','nr','nt']].copy()
for c in d.columns: d[c] = np.log1p(d[c])
h = d.index.hour + d.index.minute / 60
d['sin'] = np.sin(2*np.pi*h/24); d['cos'] = np.cos(2*np.pi*h/24)
d['lag_day']  = d['load'].shift(STEPS_DAY)
d['roll_day'] = d['lag_day'].rolling(STEPS_DAY//12, center=True, min_periods=1).mean()
d = d.dropna()

# ---------- 3. Chronological split + scaling ----------
n = len(d); a, b = int(.70*n), int(.85*n)
X = d.values.astype('float64')
mu, sd = X[:a].mean(0), X[:a].std(0) + 1e-8
Xn = (X - mu) / sd
idx = np.arange(WIN, n)
Xw = np.stack([Xn[i-WIN:i] for i in idx]); yw = Xn[idx, 0]
tr, va, te = slice(0, a-WIN), slice(a-WIN, b-WIN), slice(b-WIN, None)

# ---------- 4. LSTM ----------
model = tf.keras.Sequential([
    tf.keras.layers.Input((WIN, X.shape[1])),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)])
model.compile(tf.keras.optimizers.Adam(1e-3), loss='mse')
hist = model.fit(Xw[tr], yw[tr], validation_data=(Xw[va], yw[va]),
    epochs=60, batch_size=64, verbose=2,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=.5)])

# ---------- 5. Evaluation (original % scale) ----------
inv = lambda z: np.expm1(z * sd[0] + mu[0])
pred  = inv(model.predict(Xw[te], verbose=0).ravel())
true  = inv(yw[te])
naive = inv(Xw[te][:, -1, 0])                                   # next = last value
seasonal = np.expm1(Xn[idx, 8][te] * sd[8] + mu[8])             # same time yesterday

results = pd.DataFrame([{'Model': nm, 'R2': r2_score(true, p), 'MAE': mean_absolute_error(true, p),
                         'RMSE': mean_squared_error(true, p)**.5}
    for nm, p in [('Naive (last value)', naive), ('Seasonal naive (yesterday)', seasonal), ('LSTM', pred)]]).round(3)
print(results.to_string(index=False))
results.to_csv('results_table.csv', index=False)

# ---------- 6. Figures ----------
plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': .3})
out = pd.DataFrame({'actual_load': true, 'predicted_load': pred}, index=d.index[idx][te])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(out.index[:576], true[:576], label='Actual', lw=1.5)
ax.plot(out.index[:576], pred[:576], label='LSTM prediction', lw=1.5, alpha=.85)
ax.set_title('Total datacenter CPU load - LSTM forecast (test set, 2 days)')
ax.set_xlabel('Time'); ax.set_ylabel('CPU load (%)'); ax.legend()
fig.tight_layout(); fig.savefig('fig1_forecast.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(true, pred, s=5, alpha=.4)
lim = max(true.max(), pred.max()); ax.plot([0, lim], [0, lim], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual load (%)'); ax.set_ylabel('Predicted load (%)')
ax.set_title(f'Actual vs predicted (R2 = {r2_score(true, pred):.2f})'); ax.legend()
fig.tight_layout(); fig.savefig('fig2_scatter.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(hist.history['loss'], label='Train'); ax.plot(hist.history['val_loss'], label='Validation')
ax.set_title('LSTM training curve'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss'); ax.legend()
fig.tight_layout(); fig.savefig('fig3_loss.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(results['Model'], results['R2'], color=['#999', '#bbb', '#2a7'])
ax.axhline(0, color='k', lw=.8); ax.set_ylabel('R2 (test set)'); ax.set_title('Model comparison')
plt.xticks(rotation=12); fig.tight_layout(); fig.savefig('fig4_comparison.png', dpi=200); plt.show()

# ---------- 7. Save for the HVAC / Random Forest part ----------
out.to_csv('lstm_predictions.csv'); model.save('lstm_cpu_load.keras')
